# 2. Exploratory Data Analysis: Carbon Intensity Trends

**Project:** AI for low‑carbon energy scheduling: forecasting electricity carbon intensity and recommending cleaner time windows

**Purpose:** This notebook visualizes the patterns of grid carbon intensity. Understanding these trends is critical for identifying "low-carbon valleys" where flexible energy loads should be scheduled.

**Course Reference:** Visualization techniques (diurnal plots, scatter correlations) are adapted from **Lab 03 (Signal Processing & EDA)**. The focus on time-series decomposition of environmental signals follows the principles introduced in **Lab 04**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- Configuration ---
PROCESSED_DATA_DIR = 'data/processed/'
FIGURES_DIR = 'figures/'

if not os.path.exists(FIGURES_DIR):
    os.makedirs(FIGURES_DIR)

# Load the training set generated in Notebook 1
df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'train_ci.csv'), parse_dates=['timestamp'])
df.set_index('timestamp', inplace=True)

## 2.1 Statistical Summary

**Objective:** Describe the central tendency and spread of our target and features. (Marking Criteria: Statistical Analysis).

**Ref:** Descriptive statistics are a fundamental step from **Lab 01**.

In [ ]:
print("--- Statistical Summary of Processed Data ---")
print(df[['carbon_intensity', 'energy']].describe().to_string())

plt.figure(figsize=(10, 5))
sns.histplot(df['carbon_intensity'], kde=True, color='green')
plt.title('Distribution of Grid Carbon Intensity')
plt.xlabel('gCO2/kWh')
plt.savefig(os.path.join(FIGURES_DIR, 'intensity_distribution.png'))
plt.show()

## 2.2 Diurnal Carbon Intensity Cycle

**Insight:** This plot shows the average "cleanliness" of the grid throughout a 24-hour period. Typically, the grid is cleanest at night when demand is low and base-load renewables provide a larger share of the mix.

**Ref:** Line plots with confidence intervals are a standard EDA tool from **Lab 02**.

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='hour', y='carbon_intensity', errorbar=('ci', 95))
plt.title('Average Hourly Grid Carbon Intensity (London Proxy)')
plt.ylabel('Carbon Intensity (gCO2/kWh)')
plt.xlabel('Hour of Day')
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(FIGURES_DIR, 'hourly_carbon_intensity.png'))
plt.show()

## 2.3 Relationship between Demand and Intensity

**Concept:** The "Carbon-Demand Link". As demand increases, the grid often activates inefficient, carbon-heavy peaker plants. 

**Ref:** Correlation analysis and scatter plotting follow the methodology in **Lab 03b**.

In [ ]:
plt.figure(figsize=(10, 6))
# Sample 2000 points to avoid overplotting (Standard Data Science practice)
sns.scatterplot(data=df.sample(2000), x='energy', y='carbon_intensity', alpha=0.5)
plt.title('System Load vs. Carbon Intensity Correlation')
plt.xlabel('Aggregated Demand (kWh)')
plt.ylabel('Carbon Intensity (gCO2/kWh)')
plt.savefig(os.path.join(FIGURES_DIR, 'demand_vs_intensity_scatter.png'))
plt.show()

## 2.4 Temporal Autocorrelation

**Predictive Validation:** If intensity is highly autocorrelated (values at time *t* are similar to time *t-1*), then our lag-based features in the next notebook will be highly effective.

**Ref:** Autocorrelation plots are introduced in **Lab 04** for verifying time-series stationarity.

In [ ]:
from pandas.plotting import autocorrelation_plot

plt.figure(figsize=(12, 6))
autocorrelation_plot(df['carbon_intensity'].iloc[:1000])
plt.title('Carbon Intensity Autocorrelation (Proving Lag Predictability)')
plt.savefig(os.path.join(FIGURES_DIR, 'intensity_autocorr.png'))
plt.show()